In [2]:
import os
import time
import pickle

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.metrics import f1_score, classification_report, roc_auc_score
from transformers import AutoTokenizer, AutoModel

In [3]:
RANDOM_STATE = 42
WORK = "/kaggle/working"
os.makedirs(WORK, exist_ok=True)
CLASS_NAMES = ["access-control", "arithmetic", "other",
               "reentrancy", "safe", "unchecked-calls"]
N_CLASSES = 6
ENCODER = "microsoft/unixcoder-base"
WINDOW = 512               
N_WINDOWS = 8             
BODY = WINDOW - 2          # removing [CLS] and [SEP]
MAX_TOKENS = 64 * BODY     

N_TRAIN = 70000            # subsample: a full pass is 10-17 hours
EPOCHS = 3
BATCH = 4                  
ACCUM = 2                  
LR_ENCODER = 3e-5          
LR_HEAD = 1e-3             
WARMUP = 0.06

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

N_GPU = torch.cuda.device_count()
DEVICE = "cuda" if N_GPU else "cpu"

torch 2.11.0+cu128  gpus 2


In [4]:
train_df = pd.read_parquet("/kaggle/input/datasets/mdshagorahmed2011030/pre-processed/train_preprocessed.parquet",
                           columns=["source_capped"] + CLASS_NAMES)
val_df = pd.read_parquet("/kaggle/input/datasets/mdshagorahmed2011030/pre-processed/validation_preprocessed.parquet",
                         columns=["source_capped"] + CLASS_NAMES)
test_df = pd.read_parquet("/kaggle/input/datasets/mdshagorahmed2011030/pre-processed/test_preprocessed.parquet",
                          columns=["source_capped"] + CLASS_NAMES)

y_train_all = train_df[CLASS_NAMES].values.astype(np.float32)
y_val = val_df[CLASS_NAMES].values.astype(np.int8)
y_test = test_df[CLASS_NAMES].values.astype(np.int8)

print("   train %d, val %d, test %d" % (len(train_df), len(val_df),
                                        len(test_df)))

   train 79641, val 10861, test 15972


In [7]:
print("Tokenising ...")

tokenizer = AutoTokenizer.from_pretrained(ENCODER)
tokenizer.model_max_length = 10 ** 9      # we window it ourselves
CLS = tokenizer.cls_token_id
SEP = tokenizer.sep_token_id
PAD = tokenizer.pad_token_id


def tokenise(texts, tag):
    cache = WORK + "/tokens_" + tag + ".pkl"
    if os.path.exists(cache):
        with open(cache, "rb") as f:
            out = pickle.load(f)
        print("   %-6s loaded from cache, %d contracts" % (tag,
                                                           len(out)))
        return out

    out = []
    start = time.time()
    for i in range(0, len(texts), 128):
        batch = [("" if t is None else t) for t in texts[i:i + 128]]
        ids = tokenizer(batch, add_special_tokens=False,
                        truncation=False,
                        return_attention_mask=False)["input_ids"]
        out += [np.asarray(x[:MAX_TOKENS], dtype=np.int32) for x in ids]
        if i and i % 12800 == 0:
            print("      %s %d / %d   %d s" % (tag, i, len(texts),
                                               time.time() - start))
    with open(cache, "wb") as f:
        pickle.dump(out, f)
    print("   %-6s %d contracts in %d s" % (tag, len(out),
                                            time.time() - start))
    return out


train_ids = tokenise(train_text, "train")
val_ids = tokenise(val_text, "val")
test_ids = tokenise(test_text, "test")
del train_text, val_text, test_text

lengths = np.array([len(x) for x in train_ids])
covered = np.mean(np.minimum(lengths, N_WINDOWS * BODY)
                  / np.maximum(lengths, 1))
print("   tokens per contract: median %d, p95 %d, max %d" % (
    np.median(lengths), np.percentile(lengths, 95), lengths.max()))
print("   share of each contract inside %d windows: %.1f%%" % (
    N_WINDOWS, 100 * covered))

Tokenising ...


config.json:   0%|          | 0.00/691 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.11k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/938k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/444k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

      train 12800 / 70000   116 s
      train 25600 / 70000   225 s
      train 38400 / 70000   358 s
      train 51200 / 70000   494 s
      train 64000 / 70000   627 s
   train  70000 contracts in 688 s
   val    10861 contracts in 104 s
      test 12800 / 15972   119 s
   test   15972 contracts in 150 s
   tokens per contract: median 5184, p95 20368, max 32640
   share of each contract inside 8 windows: 68.9%


In [8]:
def window_starts(n_tokens):
    if n_tokens <= BODY:
        return [0]
    starts = list(range(0, n_tokens, BODY))
    if len(starts) > N_WINDOWS:
        pick = np.unique(np.linspace(0, len(starts) - 1, N_WINDOWS)
                         .round().astype(int))
        starts = [starts[i] for i in pick]
    return starts


def make_batch(sequences):
    starts = [window_starts(len(s)) for s in sequences]
    n_win = max(len(s) for s in starts)
    longest = 2
    for seq, st in zip(sequences, starts):
        for s in st:
            longest = max(longest, len(seq[s:s + BODY]) + 2)

    ids = np.full((len(sequences), n_win, longest), PAD, dtype=np.int64)
    mask = np.zeros((len(sequences), n_win), dtype=np.int64)
    for b, (seq, st) in enumerate(zip(sequences, starts)):
        for w, s in enumerate(st):
            piece = seq[s:s + BODY]
            ids[b, w, 0] = CLS
            ids[b, w, 1:1 + len(piece)] = piece
            ids[b, w, 1 + len(piece)] = SEP
            mask[b, w] = 1
    return torch.from_numpy(ids), torch.from_numpy(mask)

_long = np.arange(BODY * 20, dtype=np.int32)
assert len(window_starts(len(_long))) == N_WINDOWS
assert window_starts(100) == [0]
_ids, _mask = make_batch([_long, np.arange(100, dtype=np.int32)])
assert _ids.shape[0] == 2 and _ids.shape[1] == N_WINDOWS
assert _mask.tolist()[1] == [1] + [0] * (N_WINDOWS - 1)
print("   windowing self-test passed")

   windowing self-test passed


In [11]:
class WindowModel(nn.Module):
    def __init__(self, encoder, n_classes):
        super().__init__()
        self.encoder = encoder
        size = encoder.config.hidden_size
        self.attention = nn.Sequential(nn.Linear(size, 256), nn.Tanh(),
                                       nn.Linear(256, 1))
        self.dropout = nn.Dropout(0.1)
        self.head = nn.Linear(2 * size, n_classes)

    def forward(self, ids, window_mask):
        batch, n_win, n_tok = ids.shape
        flat = ids.reshape(batch * n_win, n_tok)
        real = window_mask.reshape(-1) > 0
        kept = flat[real]
        attn = (kept != PAD).long()
        hidden = self.encoder(input_ids=kept,attention_mask=attn).last_hidden_state
        m = attn.unsqueeze(-1).to(hidden.dtype)
        pooled = (hidden * m).sum(1) / m.sum(1).clamp(min=1.0)
        full = pooled.new_zeros((batch * n_win, pooled.shape[-1]))
        full[real] = pooled
        full = full.view(batch, n_win, -1)
        scores = self.attention(full).squeeze(-1).float()
        scores = scores.masked_fill(window_mask == 0, -1e4)
        weights = torch.softmax(scores, 1).to(full.dtype).unsqueeze(-1)
        by_attention = (full * weights).sum(1)
        by_max = full.masked_fill((window_mask == 0).unsqueeze(-1),-1e4).max(1).values
        return self.head(self.dropout(torch.cat([by_attention, by_max], -1)))


print("Building the model ...")
encoder = AutoModel.from_pretrained(ENCODER)          # <- this line
encoder.gradient_checkpointing_enable()
model = WindowModel(encoder, N_CLASSES).to(DEVICE)


model = WindowModel(encoder, N_CLASSES).to(DEVICE)
net = nn.DataParallel(model) if N_GPU > 1 else model
print("   encoder:", ENCODER)
print("   parameters: %.1f M" % (sum(p.numel()
                                     for p in model.parameters()) / 1e6))
positives = y_train.sum(0)
pos_weight = torch.tensor(
    np.clip((len(y_train) - positives) / np.maximum(positives, 1),
            1.0, 20.0), dtype=torch.float32, device=DEVICE)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
print("   class weights:", pos_weight.cpu().numpy().round(2))
optimizer = torch.optim.AdamW([
    {"params": model.encoder.parameters(), "lr": LR_ENCODER},
    {"params": [p for n, p in model.named_parameters()
                if not n.startswith("encoder.")], "lr": LR_HEAD},
], weight_decay=0.01)

batch_size = BATCH * max(1, N_GPU)
steps_per_epoch = int(np.ceil(len(train_ids) / batch_size))
total_steps = int(np.ceil(steps_per_epoch / ACCUM)) * EPOCHS
warmup_steps = max(1, int(WARMUP * total_steps))
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lambda s: min((s + 1) / warmup_steps,
                  max(0.0, (total_steps - s)
                      / max(1, total_steps - warmup_steps))))
scaler = torch.amp.GradScaler("cuda")

print("   batch %d, %d steps per epoch, %d epochs" % (
    batch_size, steps_per_epoch, EPOCHS))

Building the model ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

   encoder: microsoft/unixcoder-base
   parameters: 126.1 M
   class weights: [5.81 4.86 2.79 2.29 1.95 1.19]
   batch 8, 8750 steps per epoch, 3 epochs


In [13]:
def predict(sequences, tag=""):
    net.eval()
    order = np.argsort([len(s) for s in sequences])
    out = np.zeros((len(sequences), N_CLASSES), dtype=np.float32)
    size = batch_size * 2
    start = time.time()
    with torch.no_grad():
        for i in range(0, len(order), size):
            chosen = order[i:i + size]
            ids, mask = make_batch([sequences[j] for j in chosen])
            with torch.autocast("cuda", dtype=torch.float16):
                logits = net(ids.to(DEVICE), mask.to(DEVICE))
            out[chosen] = torch.sigmoid(logits.float()).cpu().numpy()
            if tag and i and i % (size * 100) == 0:
                print("      %s %d / %d   %d s" % (
                    tag, i, len(order), time.time() - start))
    return out

In [15]:
best_f1 = -1.0
best_state = None
best_val_probs = None
start_time = time.time()

for epoch in range(EPOCHS):
    net.train()
    shuffled = np.random.default_rng(RANDOM_STATE + epoch).permutation(
        len(train_ids))
    running_loss = 0.0
    n_batches = 0
    optimizer.zero_grad(set_to_none=True)

    for step in range(steps_per_epoch):
        chosen = shuffled[step * batch_size:(step + 1) * batch_size]
        if len(chosen) == 0:
            continue
        ids, mask = make_batch([train_ids[j] for j in chosen])
        targets = torch.from_numpy(y_train[chosen]).to(DEVICE)

        with torch.autocast("cuda", dtype=torch.float16):
            logits = net(ids.to(DEVICE), mask.to(DEVICE))
        loss = loss_fn(logits.float(), targets) / ACCUM
        scaler.scale(loss).backward()

        if (step + 1) % ACCUM == 0 or step == steps_per_epoch - 1:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        running_loss += float(loss.item()) * ACCUM
        n_batches += 1

        if (step + 1) % max(1, steps_per_epoch // 20) == 0:
            done = time.time() - start_time
            print("   epoch %d  step %d/%d  loss %.4f  %d s"
                  "  (epoch eta %d s)" % (
                      epoch + 1, step + 1, steps_per_epoch,
                      running_loss / n_batches, done,
                      done / (step + 1) * steps_per_epoch))

    # ---- score on validation after each epoch ----
    val_probs = predict(val_ids, tag="val")
    val_f1 = f1_score(y_val, (val_probs >= 0.5).astype(int),
                      average="macro", zero_division=0)
    print()
    print("   epoch %d done   train loss %.4f   val macro-F1 %.4f"
          % (epoch + 1, running_loss / n_batches, val_f1))

    if val_f1 > best_f1:
        best_f1 = val_f1
        best_val_probs = val_probs
        # kept on the CPU in half precision, so three epochs of
        # snapshots cost ~250 MB rather than filling the GPU
        best_state = {k: v.detach().cpu().half()
                      for k, v in model.state_dict().items()}
        print("   best so far - keeping this epoch")
        np.save(WORK + "/TF_code_val.npy",best_val_probs.astype(np.float32))
        torch.save({"state_dict": best_state, "encoder": ENCODER,
                    "windows": N_WINDOWS, "window_size": WINDOW,
                    "classes": CLASS_NAMES}, WORK + "/TF_code.pt")
        print("   checkpoint saved")

    print()

print("   trained in %.1f minutes" % ((time.time() - start_time) / 60))

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


   epoch 1  step 437/8750  loss 0.8401  479 s  (epoch eta 9607 s)
   epoch 1  step 874/8750  loss 0.7337  968 s  (epoch eta 9699 s)
   epoch 1  step 1311/8750  loss 0.6824  1457 s  (epoch eta 9725 s)
   epoch 1  step 1748/8750  loss 0.6525  1946 s  (epoch eta 9742 s)
   epoch 1  step 2185/8750  loss 0.6249  2437 s  (epoch eta 9760 s)
   epoch 1  step 2622/8750  loss 0.6009  2924 s  (epoch eta 9758 s)
   epoch 1  step 3059/8750  loss 0.5782  3413 s  (epoch eta 9763 s)
   epoch 1  step 3496/8750  loss 0.5629  3902 s  (epoch eta 9766 s)
   epoch 1  step 3933/8750  loss 0.5472  4388 s  (epoch eta 9762 s)
   epoch 1  step 4370/8750  loss 0.5332  4876 s  (epoch eta 9763 s)
   epoch 1  step 4807/8750  loss 0.5210  5364 s  (epoch eta 9764 s)
   epoch 1  step 5244/8750  loss 0.5106  5849 s  (epoch eta 9760 s)
   epoch 1  step 5681/8750  loss 0.5003  6336 s  (epoch eta 9759 s)
   epoch 1  step 6118/8750  loss 0.4911  6828 s  (epoch eta 9766 s)
   epoch 1  step 6555/8750  loss 0.4828  7319 s  (ep

In [16]:
model.load_state_dict({k: v.float()
                       for k, v in best_state.items()})
test_probs = predict(test_ids, tag="test")

test_pred = (test_probs >= 0.5).astype(int)
print("val  macro-F1 : %.4f" % best_f1)
print("test macro-F1 : %.4f" % f1_score(y_test, test_pred,
                                        average="macro",
                                        zero_division=0))
print("test micro-F1 : %.4f" % f1_score(y_test, test_pred,
                                        average="micro",
                                        zero_division=0))
print("test AUC      : %.4f" % roc_auc_score(y_test, test_probs,
                                             average="macro"))
print()
print(classification_report(y_test, test_pred,
                            target_names=CLASS_NAMES,
                            zero_division=0))

      test 1600 / 15972   14 s
      test 3200 / 15972   40 s
      test 4800 / 15972   80 s
      test 6400 / 15972   135 s
      test 8000 / 15972   191 s
      test 9600 / 15972   247 s
      test 11200 / 15972   304 s
      test 12800 / 15972   360 s
      test 14400 / 15972   416 s
val  macro-F1 : 0.8860
test macro-F1 : 0.8882
test micro-F1 : 0.9006
test AUC      : 0.9838

                 precision    recall  f1-score   support

 access-control       0.79      0.87      0.83      2346
     arithmetic       0.85      0.93      0.89      2715
          other       0.79      0.90      0.84      4209
     reentrancy       0.85      0.91      0.88      4846
           safe       0.93      0.94      0.94      5422
unchecked-calls       0.95      0.95      0.95      7289

      micro avg       0.88      0.92      0.90     26827
      macro avg       0.86      0.92      0.89     26827
   weighted avg       0.88      0.92      0.90     26827
    samples avg       0.90      0.93      0.90 